In [2]:
import pandas as pd

In [4]:
header_df=pd.read_excel(r'C:\Users\kulareddy\Downloads\New freq load\SubscriptionHeader_04012026.xlsx')

In [6]:
account_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Account-2_24_2026.csv")

C:\Users\kulareddy\AppData\Local\Temp\ipykernel_35968\3559953654.py:1: DtypeWarning: Columns (19,66) have mixed types. Specify dtype option on import or set low_memory=False.
  account_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Account-2_24_2026.csv")


In [7]:
address_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Address-2_24_2026.csv")

C:\Users\kulareddy\AppData\Local\Temp\ipykernel_35968\2637148280.py:1: DtypeWarning: Columns (19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  address_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Address-2_24_2026.csv")


In [8]:
oracle_df=pd.read_excel(r"C:\Working transformation and lookup\Reports\Convergint Oracle Customer Detailed Dev3 20260220 01.xlsx")

Stats for Customer details

In [ ]:
def normalize_key(series):
    """
    Normalize keys so Excel-looking matches actually match in pandas.
    """
    s = series.astype("string")

    s = (
        s.str.replace("\u00A0", "", regex=False)          # non-breaking space
         .str.replace(r"[\u200B-\u200D\uFEFF]", "", regex=True)  # zero-width chars
         .str.replace(r"\r|\n|\t", "", regex=True)       # line breaks / tabs
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)           # remove trailing .0
         .str.replace(r"\s+", "", regex=True)            # remove all spaces
    )

    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })

    return s


def build_lookup(df, key_col, value_col, normalize_value=False):
    """
    Creates a key -> value map.
    Keeps the first match if duplicates exist.
    """
    temp = df[[key_col, value_col]].copy()
    temp[key_col] = normalize_key(temp[key_col])

    if normalize_value:
        temp[value_col] = normalize_key(temp[value_col])

    temp = (
        temp.dropna(subset=[key_col])
            .drop_duplicates(subset=[key_col], keep="first")
    )

    return temp.set_index(key_col)[value_col]


def insert_after(df, base_col, new_col, values):
    """
    Inserts a new column immediately to the right of base_col.
    """
    pos = df.columns.get_loc(base_col) + 1
    df.insert(pos, new_col, values)


def build_match_stats(source_clean, sfid_raw, oracle_raw, row_name):
    """
    Stats format:
    Total Count
    Count Not matching for SF id
    Count Not matching for Oracle id
    """
    source_present = source_clean.notna()

    return {
        "Field": row_name,
        "Total Count": int(source_present.sum()),
        "Count Not matching for SF id": int((source_present & sfid_raw.isna()).sum()),
        "Count Not matching for Oracle id": int((source_present & sfid_raw.notna() & oracle_raw.isna()).sum())
    }


# -------------------------------------------------
# 0. Normalize source columns used for matching
# -------------------------------------------------
primary_source_clean = normalize_key(header_df["PrimaryPartyId"])
billtoacct_source_clean = normalize_key(header_df["BillToAccountId"])
billtosite_source_clean = normalize_key(header_df["BillToSiteUseId"])


# -----------------------------
# 1. Build all required lookups
# -----------------------------

# account_df lookups
account_legacy_to_sfid = build_lookup(
    account_df,
    "CVG_Legacy_ID__c",
    "Id",
    normalize_value=True
)

# address_df lookup
address_legacy_to_sfid = build_lookup(
    address_df,
    "CVG_Legacy_ID__c",
    "Id",
    normalize_value=True
)

# oracle_df lookups
orig_sys_ref_to_party_id = build_lookup(
    oracle_df,
    "ORIG_SYS_REF",
    "PARTY_ID",
    normalize_value=False
)

orig_sys_ref_to_account_id = build_lookup(
    oracle_df,
    "ORIG_SYS_REF",
    "ACCOUNT_ID",
    normalize_value=False
)

site_orig_sys_ref_to_site_use_id = build_lookup(
    oracle_df,
    "SITE_ORIG_SYS_REF",
    "SITE_USE_ID",
    normalize_value=False
)


# -----------------------------------
# 2. Create mapped raw values
# -----------------------------------

# PrimaryPartyId
primary_sfid_raw = primary_source_clean.map(account_legacy_to_sfid)
primary_oracle_raw = primary_sfid_raw.map(orig_sys_ref_to_party_id)

# BillToAccountId
billtoacct_sfid_raw = billtoacct_source_clean.map(account_legacy_to_sfid)
billtoacct_oracle_raw = billtoacct_sfid_raw.map(orig_sys_ref_to_account_id)

# BillToSiteUseId
billtosite_sfid_raw = billtosite_source_clean.map(address_legacy_to_sfid)
billtosite_oracle_raw = billtosite_sfid_raw.map(site_orig_sys_ref_to_site_use_id)


# -----------------------------------
# 3. Replace non-matches with #N/A
# -----------------------------------
primary_sfid = primary_sfid_raw.fillna("#N/A")
primary_oracle = primary_oracle_raw.fillna("#N/A")

billtoacct_sfid = billtoacct_sfid_raw.fillna("#N/A")
billtoacct_oracle = billtoacct_oracle_raw.fillna("#N/A")

billtosite_sfid = billtosite_sfid_raw.fillna("#N/A")
billtosite_oracle = billtosite_oracle_raw.fillna("#N/A")


# -----------------------------------
# 4. Create header_inprogress output
# -----------------------------------
header_inprogress = header_df.copy()

insert_after(header_inprogress, "PrimaryPartyId", "PrimaryPartyId_SFId", primary_sfid)
insert_after(header_inprogress, "PrimaryPartyId_SFId", "PrimaryPartyId_OracleId", primary_oracle)

insert_after(header_inprogress, "BillToAccountId", "BillToAccountId_SFId", billtoacct_sfid)
insert_after(header_inprogress, "BillToAccountId_SFId", "BillToAccountId_OracleId", billtoacct_oracle)

insert_after(header_inprogress, "BillToSiteUseId", "BillToSiteUseId_SFId", billtosite_sfid)
insert_after(header_inprogress, "BillToSiteUseId_SFId", "BillToSiteUseId_OracleId", billtosite_oracle)


# -----------------------------------
# 5. Create header_transformed output
# -----------------------------------
header_transformed = header_df.copy()

header_transformed["PrimaryPartyId"] = primary_oracle
header_transformed["BillToAccountId"] = billtoacct_oracle
header_transformed["BillToSiteUseId"] = billtosite_oracle


# -----------------------------------
# 6. Create stats table
# -----------------------------------
stats_data = [
    build_match_stats(
        primary_source_clean,
        primary_sfid_raw,
        primary_oracle_raw,
        "PrimaryPartyId"
    ),
    build_match_stats(
        billtoacct_source_clean,
        billtoacct_sfid_raw,
        billtoacct_oracle_raw,
        "BillToAccountId"
    ),
    build_match_stats(
        billtosite_source_clean,
        billtosite_sfid_raw,
        billtosite_oracle_raw,
        "BillToSiteUseId"
    )
]

stats_df = pd.DataFrame(stats_data)


# -----------------------------------
# 7. Save outputs
# -----------------------------------
with pd.ExcelWriter("header_inprogress.xlsx", engine="openpyxl") as writer:
    header_inprogress.to_excel(writer, sheet_name="header_inprogress", index=False)
    stats_df.to_excel(writer, sheet_name="stats", index=False)

header_transformed.to_excel("header_transformed.xlsx", index=False)


# -----------------------------------
# 8. Validation prints
# -----------------------------------
print("header_inprogress.xlsx created")
print("header_transformed.xlsx created")

print("\nShapes:")
print("header_df:", header_df.shape)
print("header_inprogress:", header_inprogress.shape)
print("header_transformed:", header_transformed.shape)




header_inprogress.xlsx created
header_transformed.xlsx created

Shapes:
header_df: (3230, 25)
header_inprogress: (3230, 31)
header_transformed: (3230, 25)

Stats:
             Field  Total Count  Count Not matching for SF id  \
0   PrimaryPartyId         3230                           217   
1  BillToAccountId         3229                           147   
2  BillToSiteUseId         3229                           726   

   Count Not matching for Oracle id  
0                                33  
1                                38  
2                                36  
header_stats.xlsx created


In [ ]:
print("\nStats:")
print(stats_df)

#stats_df.to_excel("header_stats.xlsx", index=False)
#print("header_stats.xlsx created")